In [4]:
from pathlib import Path
import json, pandas as pd, numpy as np

DATA = Path('./data/')

schema = json.loads((DATA / 'feature_schema_context.json').read_text(encoding='utf-8'))
df = pd.read_parquet(DATA / 'model_features_context.parquet')
print(df.shape)                            # (2629440, 56)

# pd.set_option("display.max_columns", None)
# df
# 
INPUTS = schema['inputs']          # 입력 39개 (직접 고르지 말고 이 목록을 쓰기)
TARGET = 'rentals_target_1h'       # 반납 모델은 'returns_target_1h'

use = df.coverage_status.eq('observed_operating') & df[TARGET].notna()
train = df[use & df.split.eq('train')]
valid = df[use & df.split.eq('validation')]
test  = df[use & df.split.eq('test')]      # 마지막까지 열어보지 않기
print(len(train), len(valid), len(test))

(2629440, 56)
1569378 526178 524875


In [ ]:
import json
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

# =====================================================================
# STEP 2: Splitting Data into Train, Validation, and Test
# =====================================================================
print("📊 Executing Step 2: Creating time-series splitting masks...")

INPUTS = schema['inputs']  # Evaluates the exact 39 fields from the JSON
TARGETS = ['rentals_target_1h', 'returns_target_1h'] 

# Strict filter condition matching your project constraints
use = df.coverage_status.eq('observed_operating') & df['rentals_target_1h'].notna() & df['returns_target_1h'].notna()

train_mask = use & df.split.eq('train')
valid_mask = use & df.split.eq('validation')
test_mask = use & df.split.eq('test')

# Print split records to verify row counts match (1569378, 526178, 524875)
print(f"   ↳ Train samples: {train_mask.sum():,}")
print(f"   ↳ Validation samples: {valid_mask.sum():,}")
print(f"   ↳ Test samples: {test_mask.sum():,}")

# Initialize collectors for Step 5 deliverables
metrics_records = []
pred_df_list = []

# Score evaluation function 
def score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None) # Rule 5: Clip negative values to 0
    err = y_pred - y_true
    total = np.abs(y_true).sum()
    
    mae = np.abs(err).mean()
    rmse = np.sqrt((err**2).mean())
    wape = np.abs(err).sum() / total if total else np.nan
    bias = err.mean()
    
    return {'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'Bias': bias}

# =====================================================================
# STEP 3 & 4: Model Training and Validation Loops
# =====================================================================
for target in TARGETS:
    print(f"\n🚀 Training LightGBM Model for target variable: {target}...")
    
    # Isolate training subsets securely using the boolean masks
    X_train, y_train = df.loc[train_mask, INPUTS].copy(), df.loc[train_mask, target]
    X_valid, y_valid = df.loc[valid_mask, INPUTS].copy(), df.loc[valid_mask, target]
    X_test, y_test   = df.loc[test_mask, INPUTS].copy(), df.loc[test_mask, target]
    
    # Explicit conversion of textual elements into categoricals for LightGBM support
    categorical_cols = ['station_group', 'region', 'commute_period', 'holiday_type']
    for col in categorical_cols:
        if col in X_train.columns:
            X_train[col] = X_train[col].astype('category')
            X_valid[col] = X_valid[col].astype('category')
            X_test[col] = X_test[col].astype('category')
            
    # Configure LightGBM Regressor matching Team Rule 6 (Fixed random state)
    model = lgb.LGBMRegressor(
        objective='regression',
        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42, 
        verbosity=-1
    )
    
    # Train the model trackable by Early Stopping
    start_time = time.time()
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    train_seconds = time.time() - start_time
    print(f"⏱️ Model training completed in {train_seconds:.2f} seconds.")
    joblib.dump(model, f'./models/bicycle_lightgbm_{target}.pkl')
    
    # Generate predictions across splits (preserving matching index positioning)
    valid_pred_arr = np.clip(model.predict(X_valid), 0, None)
    test_pred_arr  = np.clip(model.predict(X_test), 0, None)
    
    valid_pred_series = pd.Series(valid_pred_arr, index=X_valid.index)
    test_pred_series  = pd.Series(test_pred_arr, index=X_test.index)
    
    # Mapped directly to your actual verified data strings ('역세권', 'morning', etc.)
    segments = {
        '전체': (X_valid.index, X_test.index),
        
        '출근': (X_valid.index[df.loc[X_valid.index, 'commute_period'] == 'morning'], 
                X_test.index[df.loc[X_test.index, 'commute_period'] == 'morning']),
        
        '퇴근': (X_valid.index[df.loc[X_valid.index, 'commute_period'] == 'evening'], 
                X_test.index[df.loc[X_test.index, 'commute_period'] == 'evening']),
        
        '역세권': (X_valid.index[df.loc[X_valid.index, 'station_group'] == '역세권'], 
                  X_test.index[df.loc[X_test.index, 'station_group'] == '역세권']),
        
        '업무지구': (X_valid.index[df.loc[X_valid.index, 'station_group'] == '업무지구'], 
                    X_test.index[df.loc[X_test.index, 'station_group'] == '업무지구']),
        
        '주거지': (X_valid.index[df.loc[X_valid.index, 'station_group'] == '주거지'], 
                  X_test.index[df.loc[X_test.index, 'station_group'] == '주거지'])
    }
    
    for seg_name, (v_idx, t_idx) in segments.items():
        # A. Evaluate Validation split
        y_val_true_seg = df.loc[v_idx, target]
        y_val_pred_seg = valid_pred_series.loc[v_idx]
        v_metrics = score(y_val_true_seg, y_val_pred_seg)
        
        metrics_records.append({
            'model': 'LightGBM', 'target': target, 'split': 'validation', 'segment': seg_name,
            'n': len(y_val_true_seg), 'MAE': v_metrics['MAE'], 'RMSE': v_metrics['RMSE'],
            'WAPE': v_metrics['WAPE'], 'Bias': v_metrics['Bias'], 'train_seconds': train_seconds
        })
        
        # B. Evaluate Test split (2025 Test Evaluated once)
        y_test_true_seg = df.loc[t_idx, target]
        y_test_pred_seg = test_pred_series.loc[t_idx]
        t_metrics = score(y_test_true_seg, y_test_pred_seg)
        
        metrics_records.append({
            'model': 'LightGBM', 'target': target, 'split': 'test', 'segment': seg_name,
            'n': len(y_test_true_seg), 'MAE': t_metrics['MAE'], 'RMSE': t_metrics['RMSE'],
            'WAPE': t_metrics['WAPE'], 'Bias': t_metrics['Bias'], 'train_seconds': train_seconds
        })

    # Collect row-level target mappings for the required pred Parquet file
    for split_name, mask, pred_vals in [('validation', valid_mask, valid_pred_arr), ('test', test_mask, test_pred_arr)]:
        sub_pred = df.loc[mask, ['station_id', 'timestamp']].copy()
        sub_pred['split'] = split_name
        sub_pred['target'] = target
        sub_pred['y_true'] = df.loc[mask, target].astype(float)
        sub_pred['y_pred'] = pred_vals.astype(float)
        pred_df_list.append(sub_pred)

# =====================================================================
# STEP 5: Exporting Results in Strict Formatting Forms
# =====================================================================
print("\n💾 Writing production outputs to Shared Drive directory path...")

DATA = Path('./results/')

metrics_df = pd.DataFrame(metrics_records)
metrics_df.to_csv(DATA / 'metrics_LightGBM.csv', index=False)

pred_df = pd.concat(pred_df_list, ignore_index=True)
pred_df.to_parquet(DATA / 'pred_LightGBM.parquet', index=False, engine='pyarrow')

params_dict = {
    "inputs": INPUTS,
    "hyperparameters": {
        "objective": "regression",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "random_state": 42
    }
}
with open(DATA / 'params_LightGBM.json', 'w', encoding='utf-8') as f:
    json.dump(params_dict, f, indent=2, ensure_ascii=False)

print("🎯 Finished successfully! Step 2 & 3 pipeline executed cleanly.")


📊 Executing Step 2: Creating time-series splitting masks...
   ↳ Train samples: 1,569,378
   ↳ Validation samples: 526,178
   ↳ Test samples: 524,875

🚀 Training LightGBM Model for target variable: rentals_target_1h...


d:\jsm\ringring\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


⏱️ Model training completed in 85.37 seconds.

🚀 Training LightGBM Model for target variable: returns_target_1h...


d:\jsm\ringring\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


⏱️ Model training completed in 99.43 seconds.

💾 Writing production outputs to Shared Drive directory path...
🎯 Finished successfully! Step 2 & 3 pipeline executed cleanly.


In [4]:
import pandas as pd

# 1. Load the generated metrics file from your DATA path
metrics_path = DATA / 'metrics_LightGBM.csv'
metrics_df = pd.read_csv(metrics_path)

# 2. Separate Validation and Test for clean viewing
val_summary = metrics_df[metrics_df['split'] == 'validation'].copy()
test_summary = metrics_df[metrics_df['split'] == 'test'].copy()

# 3. Display Validation Dashboard (Use this to compare against Team A's Baseline)
print("=================== 📊 LightGBM 검증 ===================")
val_pivot = val_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(val_pivot.round(4))

# 4. Display Final Test Dashboard (The 2025 Evaluation - Checked Once)
print("\n=================== 🏁 LightGBM 테스트 ===================")
test_pivot = test_summary.pivot(index='segment', columns='target', values=['MAE', 'WAPE', 'Bias'])
display(test_pivot.round(4))


=================== 📊 LightGBM 검증 ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.3022            1.3368            0.4510   
역세권                1.6830            1.6854            0.3868   
전체                 1.3865            1.3964            0.4378   
주거지                1.1742            1.1668            0.5191   
출근                 1.6495            1.7007            0.3888   
퇴근                 2.0967            2.0770            0.3464   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.4408            0.0358           -0.0031  
역세권                0.3780            0.0956            0.0391  
전체                 0.4282            0.0432            0.0174  
주거지                0.5093           -0.0018            0.0164  
출근                 0.3416           -0.0722           -0.0664  
퇴근                 0.3459            0.1423            0.0507


=================== 🏁 LightGBM 테스트 ===================


MAE                                WAPE  \
target  rentals_target_1h returns_target_1h rentals_target_1h   
segment                                                         
업무지구               1.1884            1.2090            0.4963   
역세권                1.5126            1.5226            0.4315   
전체                 1.2577            1.2668            0.4833   
주거지                1.0727            1.0691            0.5619   
출근                 1.5011            1.5593            0.4129   
퇴근                 1.9148            1.8751            0.3919   

                                       Bias                    
target  returns_target_1h rentals_target_1h returns_target_1h  
segment                                                        
업무지구               0.4783            0.1803            0.1241  
역세권                0.4205            0.2258            0.1498  
전체                 0.4713            0.1628            0.1274  
주거지                0.5577            0.0826            0.1084  
출근                 0.3577            0.0587            0.0569  
퇴근                 0.3962            0.4158            0.3323